In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np

from optimal_long_short.model_params import KouParams
from optimal_long_short.market_params import MarketParams
from optimal_long_short.strategy import UnitExposureLongShortStrategy
from optimal_long_short.inversion import TalbotInverter, StehfestInverter, DeHoogInverter
from optimal_long_short.moments import ConditionalMoments
from optimal_long_short.monte_carlo import MonteCarlo

In [2]:
# --- model and strategy parameters ---
params = KouParams(
    mu1=-0.05, sigma1=0.20, lam1=1.0, p1=0.5, eta1_pos=1/10.1, eta1_neg=1/10.0,
    mu2=0.00, sigma2=0.15, lam2=1.0, p2=0.5, eta2_pos=1/10.1, eta2_neg=1/10.0,
    rho=0.3,
)
market = MarketParams(b=0.8, S10=1.0, S20=1.0)
strategy = UnitExposureLongShortStrategy(h0=0.5, market=market, T=1)

print(f"h0 = {strategy.h0},  T = {strategy.T}")
print(f"w1 = {strategy.w1:.6f},  w2 = {strategy.w2:.6f}")
print(f"b  = {market.b}")

h0 = 0.5,  T = 1
w1 = 1.942594,  w2 = 0.942594
b  = 0.8


In [3]:
# --- compare three inverters ---
inverters = {
    "Talbot (M=32)":   TalbotInverter(M=32),
    "Stehfest (N=12)": StehfestInverter(N=12),
    "de Hoog (M=24)":  DeHoogInverter(M=24),
}

print(f"{'Inverter':<20} {'p_surv':>10} {'E[Pi|surv]':>14} {'Var(Pi|surv)':>16}")
print("-" * 64)
for name, inv in inverters.items():
    cm = ConditionalMoments(params=params, strategy=strategy, inverter=inv)
    ps  = cm.p_surv()
    mu  = cm.conditional_mean()
    var = cm.conditional_variance()
    print(f"{name:<20} {ps:>10.6f} {mu:>14.6f} {var:>16.6f}")

Inverter                 p_surv     E[Pi|surv]     Var(Pi|surv)
----------------------------------------------------------------
Talbot (M=32)          0.900636       0.980780         0.183774
Stehfest (N=12)        0.900558       0.980841         0.183750
de Hoog (M=24)       625.069915       0.898706         0.279531


/Users/francis/Defi_interestRate/notebooks/../optimal_long_short/inversion.py:287: RuntimeWarning: invalid value encountered in divide
  q[:-2, r] = q[1:-1, r - 1] * e[1:-1, r] / e[:-2, r]


In [4]:
# --- sensitivity to h0 (Talbot) ---
import numpy as np

h0_grid = np.linspace(0.1, 2.0, 20)
inv = TalbotInverter(M=32)

p_survs, means, variances = [], [], []
for h0 in h0_grid:
    strat = UnitExposureLongShortStrategy(h0=float(h0), market=market, T=1.0)
    cm = ConditionalMoments(params=params, strategy=strat, inverter=inv)
    p_survs.append(cm.p_surv())
    means.append(cm.conditional_mean())
    variances.append(cm.conditional_variance())

print(f"{'h0':>6} {'p_surv':>10} {'E[Pi|surv]':>14} {'Var(Pi|surv)':>16}")
print("-" * 50)
for h0, ps, mu, var in zip(h0_grid, p_survs, means, variances):
    print(f"{h0:>6.2f} {ps:>10.6f} {mu:>14.6f} {var:>16.6f}")

    h0     p_surv     E[Pi|surv]     Var(Pi|surv)
--------------------------------------------------
  0.10   0.263796       1.709104         0.576983
  0.20   0.497635       1.337140         0.356155
  0.30   0.685115       1.142331         0.262291
  0.40   0.817467       1.036907         0.213729
  0.50   0.900636       0.980780         0.183774
  0.60   0.948286       0.952379         0.162326
  0.70   0.973873       0.939189         0.145436
  0.80   0.987061       0.933980         0.131563
  0.90   0.993688       0.932741         0.120034
  1.00   0.996960       0.933347         0.110451
  1.10   0.998552       0.934715         0.102500
  1.20   0.999318       0.936314         0.095908
  1.30   0.999681       0.937899         0.090434
  1.40   0.999852       0.939369         0.085871
  1.50   0.999932       0.940690         0.082049
  1.60   0.999969       0.941861         0.078830
  1.70   0.999986       0.942894         0.076101
  1.80   0.999994       0.943803         0.073774

In [5]:
# ── Test 1 : k=1..4 Laplace vs MC (generic case, no degeneracy) ──────────────
#
# Phase rates for these params:
#   r1_neg = 1/eta1_neg = 10
#   r2_neg^(k) = 1/eta2_pos - k ≈ 10.1 - k
#
#   k=1: r2_neg ≈ 9.1   k=2: r2_neg ≈ 8.1
#   k=3: r2_neg ≈ 7.1   k=4: r2_neg ≈ 6.1   <- all differ from r1_neg=10

N_PATHS = 300_000
N_STEPS = 252
SEED    = 0

inv = TalbotInverter(M=36)
cm  = ConditionalMoments(params=params, strategy=strategy, inverter=inv)

mc     = MonteCarlo(params=params, strategy=strategy,
                    n_paths=N_PATHS, n_steps=N_STEPS, seed=SEED)
result = mc.run()
surv_pay = result.payoff[result.survived]

print(f"n_survived = {result.survived.sum():,} / {N_PATHS:,}  "
      f"(p_surv  MC={result.p_surv:.6f}  Lap={cm.p_surv():.6f})")
print()
print(f"  {'k':>2}  {'Laplace':>12}  {'MC':>12}  {'abs err':>10}  {'rel err':>10}")
print("  " + "-" * 52)

laplace_moments = [cm.conditional_moment(k) for k in range(1, 5)]
mc_moments      = [float(np.mean(surv_pay**k)) for k in range(1, 5)]

for k, (lap, mc_val) in enumerate(zip(laplace_moments, mc_moments), start=1):
    print(f"  {k:>2}  {lap:>12.6f}  {mc_val:>12.6f}"
          f"  {abs(lap-mc_val):>10.2e}  {abs(lap/mc_val-1):>10.2e}")

n_survived = 271,403 / 300,000  (p_surv  MC=0.904677  Lap=0.900636)

   k       Laplace            MC     abs err     rel err
  ----------------------------------------------------
   1      0.980780      0.976774    4.01e-03    4.10e-03
   2      1.145703      1.137507    8.20e-03    7.21e-03
   3      1.592352      1.574557    1.78e-02    1.13e-02
   4      2.673813      2.621353    5.25e-02    2.00e-02


In [6]:
# ── Test 2 : consistency — conditional_moment(k) matches existing helpers ─────
#
# conditional_moment(1) must equal conditional_mean()
# conditional_moment(2) must equal conditional_second_moment()

tol_exact = 1e-12

mu1_direct = cm.conditional_mean()
mu2_direct = cm.conditional_second_moment()
mu1_general = cm.conditional_moment(1)
mu2_general = cm.conditional_moment(2)

err1 = abs(mu1_general - mu1_direct)
err2 = abs(mu2_general - mu2_direct)

print(f"conditional_moment(1) vs conditional_mean():          err = {err1:.2e}")
print(f"conditional_moment(2) vs conditional_second_moment(): err = {err2:.2e}")
print()

assert err1 < tol_exact, f"k=1 mismatch: {err1:.2e}"
assert err2 < tol_exact, f"k=2 mismatch: {err2:.2e}"
print("PASS  (both errors < 1e-12)")

conditional_moment(1) vs conditional_mean():          err = 0.00e+00
conditional_moment(2) vs conditional_second_moment(): err = 0.00e+00

PASS  (both errors < 1e-12)


In [7]:
# ── Test 3 : Laplace vs MC tolerance assertions ───────────────────────────────
#
# Tolerances account for MC noise at 300k paths.  Higher moments have more
# variance, so we allow progressively looser relative error.
#
# Relative SE of the k-th sample moment ~ std(X^k) / (E[X^k] * sqrt(n_surv)).
# With n_surv ≈ 270k the MC estimates are tight even for k=4.

tol = {1: 5e-3, 2: 1e-2, 3: 2e-2, 4: 5e-2}

failures = []
for k, (lap, mc_val) in enumerate(zip(laplace_moments, mc_moments), start=1):
    rel_err = abs(lap / mc_val - 1)
    status  = "PASS" if rel_err < tol[k] else "FAIL"
    if status == "FAIL":
        failures.append(k)
    print(f"  k={k}  rel_err={rel_err:.2e}  tol={tol[k]:.0e}  {status}")

print()
if failures:
    raise AssertionError(f"Moment(s) failed tolerance check: k={failures}")
print("ALL PASS")

  k=1  rel_err=4.10e-03  tol=5e-03  PASS
  k=2  rel_err=7.21e-03  tol=1e-02  PASS
  k=3  rel_err=1.13e-02  tol=2e-02  PASS
  k=4  rel_err=2.00e-02  tol=5e-02  PASS

ALL PASS


In [ ]:
# ── Test 4 : degenerate case — k=4 with r1_neg = r2_neg ──────────────────────
#
# eta1_neg = 0.125  =>  r1_neg = 8
# eta2_pos = 1/12   =>  r2_neg^(k) = 12 - k
#
# At k=4: r2_neg = 12 - 4 = 8 = r1_neg  (exact degeneracy)
# The 3x3 barrier matrix has identical rows 1 and 2 -> would be singular.
# Our fix: detect degeneracy, strip the spurious root, solve a 2x2 system.

from optimal_long_short.kou_model import KouZTiltedDynamics

params_degen = KouParams(
    mu1=0.05,  sigma1=0.20, lam1=1.0, p1=0.5, eta1_pos=0.10,  eta1_neg=0.125,
    mu2=0.03,  sigma2=0.15, lam2=0.8, p2=0.5, eta2_pos=1/12,  eta2_neg=1/9,
    rho=0.95,
)
market_degen   = MarketParams(b=0.8, S10=1.0, S20=1.0)
strategy_degen = UnitExposureLongShortStrategy(h0=0.5, market=market_degen, T=1.0)

# Confirm the degeneracy at k=4
dyn4 = KouZTiltedDynamics(params=params_degen, k=4)
print(f"r1_neg       = {dyn4.r1_neg:.6f}")
print(f"r2_neg^(4)   = {dyn4.r2_neg:.6f}")
print(f"degenerate   = {abs(dyn4.r1_neg - dyn4.r2_neg) < 1e-8}")
print()

N_PATHS_D = 300_000
N_STEPS_D = 252
SEED_D    = 42

inv_d = TalbotInverter(M=36)
cm_d  = ConditionalMoments(params=params_degen, strategy=strategy_degen, inverter=inv_d)

mc_d     = MonteCarlo(params=params_degen, strategy=strategy_degen,
                      n_paths=N_PATHS_D, n_steps=N_STEPS_D, seed=SEED_D)
result_d = mc_d.run()
surv_pay_d = result_d.payoff[result_d.survived]

print(f"n_survived = {result_d.survived.sum():,} / {N_PATHS_D:,}  "
      f"(p_surv  MC={result_d.p_surv:.6f}  Lap={cm_d.p_surv():.6f})")
print()
print(f"  {'k':>2}  {'Laplace':>12}  {'MC':>12}  {'abs err':>10}  {'rel err':>10}")
print("  " + "-" * 52)

laplace_d = [cm_d.conditional_moment(k) for k in range(1, 5)]
mc_d_vals = [float(np.mean(surv_pay_d**k)) for k in range(1, 5)]

for k, (lap, mc_val) in enumerate(zip(laplace_d, mc_d_vals), start=1):
    print(f"  {k:>2}  {lap:>12.6f}  {mc_val:>12.6f}"
          f"  {abs(lap-mc_val):>10.2e}  {abs(lap/mc_val-1):>10.2e}")

print()
tol_degen = {1: 5e-3, 2: 1e-2, 3: 2e-2, 4: 5e-2}
failures_d = []
for k, (lap, mc_val) in enumerate(zip(laplace_d, mc_d_vals), start=1):
    rel_err = abs(lap / mc_val - 1)
    status  = "PASS" if rel_err < tol_degen[k] else "FAIL"
    if status == "FAIL":
        failures_d.append(k)
    print(f"  k={k}  rel_err={rel_err:.2e}  tol={tol_degen[k]:.0e}  {status}")

print()
if failures_d:
    raise AssertionError(f"Degenerate moment(s) failed tolerance check: k={failures_d}")
print("ALL PASS  (degenerate 2x2 system correct)")